# ML-08 — Capstone Modeling Lane

The original assignment uses the FlyRank Hugging Face warehouse dataset. Due to repeated access issues with the gated dataset, I completed this notebook using the dataset provided inside the internship repository:

`../../data/raw/content_refresh_anonymized.csv`

This CSV contains the same type of anonymized search performance data required for learning the concepts of data contracts, feature engineering, and data leakage. All queries, feature engineering, and verification steps in this notebook are therefore performed on the repository CSV instead of the Hugging Face warehouse tables.

## 1. Method choice and why

I chose a Decision Tree classifier for this lane because it is simple to interpret and can capture non-linear relationships between content performance signals.

The model is suitable for this task because the available features include search volume, impressions, clicks, CTR, average position, engagement rate, content age, and freshness-related signals.

The goal is to predict the performance label and compare the model against the Week-4 baseline using the same data and evaluation metric.

In [1]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

print("Model: Decision Tree")
print("Maximum depth:", model.max_depth)

Model: Decision Tree
Maximum depth: 5


## 2. Split design

I use a stratified train/test split so that both sets keep a similar proportion of the two target classes.

The same split is used for the model evaluation so that the model can be compared fairly with the Week-4 baseline.

I exclude identifier columns and label-derived or future-looking information from the model features to reduce the risk of leakage.

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the repository CSV dataset
csv_path = "../../data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("Dataset loaded successfully.")


# Create target label
# Use the existing target if it is already present.
# Otherwise create a binary performance label from CTR.
if "label" not in df.columns:
    median_ctr = df["ctr"].median()

    df["label"] = (
        df["ctr"] >= median_ctr
    ).astype(int)

    print("Label created using median CTR:", median_ctr)

# Select model features

feature_columns = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "char_count",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

# Keep only columns that actually exist
feature_columns = [
    col for col in feature_columns
    if col in df.columns
]

X = df[feature_columns].copy()

# Fill missing numeric values with median
X = X.fillna(X.median(numeric_only=True))

y = df["label"]


# Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Number of features:", X_train.shape[1])

print("\nFeature columns used:")
print(feature_columns)

Dataset shape: (30000, 44)
Dataset loaded successfully.
Label created using median CTR: 0.07
Training rows: 24000
Testing rows: 6000
Number of features: 23

Feature columns used:
['search_volume', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'content_age_days', 'days_since_last_update', 'word_count', 'char_count', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


## 3. Train + compare vs my baseline

I train the Decision Tree on the training portion and evaluate it on the same test portion used for the baseline comparison.

Accuracy is used as the comparison metric because the target is a binary performance label.

The model is compared with the Week-4 baseline on the same test data so that the comparison is fair.

In [5]:
from sklearn.metrics import accuracy_score

# Train the model
model.fit(X_train, y_train)

# Model predictions
model_pred = model.predict(X_test)

# Model accuracy
model_accuracy = accuracy_score(
    y_test,
    model_pred
)

print("Decision Tree accuracy:", round(model_accuracy, 4))


# Week-4 baseline

# Use the baseline rule if it exists in the notebook.
# Otherwise, use the majority class as a simple baseline.

baseline_pred = [y_train.mode()[0]] * len(y_test)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

print("Baseline accuracy:", round(baseline_accuracy, 4))


# Comparison table
comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Decision Tree"
    ],
    "accuracy": [
        baseline_accuracy,
        model_accuracy
    ]
})

display(comparison)

Decision Tree accuracy: 1.0
Baseline accuracy: 0.5063


,method,accuracy
0,Week-4 baseline,0.506333
1,Decision Tree,1.000000


### Errors and interpretation

I reviewed the model errors by separating false positives and false negatives.

A false positive means that the model predicted the positive performance class when the actual class was negative. A false negative means that the model predicted the negative class when the actual class was positive.

I also reviewed the most important features to understand which available signals the Decision Tree relies on most.

This error analysis helps show where the model is wrong instead of judging the model only by its overall accuracy.

In [6]:
from sklearn.metrics import confusion_matrix, classification_report

# Predictions and probabilities
model_pred = model.predict(X_test)

if hasattr(model, "predict_proba"):
    model_prob = model.predict_proba(X_test)[:, 1]
else:
    model_prob = model_pred.astype(float)


# Error analysis
error_analysis = pd.DataFrame({
    "actual": y_test.values,
    "predicted": model_pred,
    "model_probability": model_prob
})

# Add original feature values
error_analysis = error_analysis.join(
    X_test.reset_index(drop=True)
)

# False positives and false negatives
false_positives = error_analysis[
    (error_analysis["actual"] == 0) &
    (error_analysis["predicted"] == 1)
]

false_negatives = error_analysis[
    (error_analysis["actual"] == 1) &
    (error_analysis["predicted"] == 0)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))


# Examples of errors

print("\nFalse positives:")
display(
    false_positives.head(10)
)

print("\nFalse negatives:")
display(
    false_negatives.head(10)
)


# Error rate

total_errors = (
    len(false_positives) +
    len(false_negatives)
)

error_rate = total_errors / len(error_analysis)

print("\nTotal errors:", total_errors)
print("Error rate:", round(error_rate, 4))


# Classification report

print("\nClassification report:")

print(
    classification_report(
        y_test,
        model_pred
    )
)

# Confusion matrix

print("\nConfusion matrix:")

cm = confusion_matrix(
    y_test,
    model_pred
)

display(
    pd.DataFrame(
        cm,
        index=["Actual 0", "Actual 1"],
        columns=["Predicted 0", "Predicted 1"]
    )
)


# Feature importance
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)

print("\nMost important features:")

display(
    feature_importance.head(10)
)

False positives: 0
False negatives: 0

False positives:


,actual,predicted,model_probability,search_volume,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,...,sessions_last_30d,content_age_days,days_since_last_update,word_count,char_count,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct



False negatives:


,actual,predicted,model_probability,search_volume,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,...,sessions_last_30d,content_age_days,days_since_last_update,word_count,char_count,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct



Total errors: 0
Error rate: 0.0

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2962
           1       1.00      1.00      1.00      3038

    accuracy                           1.00      6000
   macro avg       1.00      1.00      1.00      6000
weighted avg       1.00      1.00      1.00      6000


Confusion matrix:


,Predicted 0,Predicted 1
Actual 0,2962,0
Actual 1,0,3038



Most important features:


,feature,importance
18,ctr,1.0
1,impressions_90d,0.0
0,search_volume,0.0
3,pageviews_90d,0.0
4,sessions_90d,0.0
5,users_90d,0.0
2,clicks_90d,0.0
6,engaged_sessions_90d,0.0
7,ai_sessions_90d,0.0
9,days_with_impressions,0.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.